In [ ]:
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
from dataclasses import dataclass
import numpy as np
from statsmodels.tsa.regime_switching.markov_regression import MarkovRegression


In [8]:
data = yf.download('BTC-USD', start='2020-01-01', end='2024-01-01')
data.index = pd.to_datetime(data.index)
ddf = data['Close']
df


/var/folders/yk/03kntj1x0gxg37_rhl87gj800000gn/T/ipykernel_1770/2656553568.py:1: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download('BTC-USD', start='2020-01-01', end='2024-01-01')
[*********************100%***********************]  1 of 1 completed


Ticker,BTC-USD
Date,
2020-01-01,7200.174316
2020-01-02,6985.470215
2020-01-03,7344.884277
2020-01-04,7410.656738
2020-01-05,7411.317383
...,...
2023-12-27,43442.855469
2023-12-28,42627.855469
2023-12-29,42099.402344


In [ ]:
#Regimes that we are targeting
RegimeState = str
REGIME_TO_SIDE: dict[RegimeState, str] = {
    "bull": "UP",
    "neutral": "NO_TRADE",
    "bear": "DOWN",
}

In [ ]:
#Model summary of how we are going to execute these trades
#P(S_t = j | F_t): Probability of being in regime j at time t given all information up to time t
@dataclass(frozen=True)
class RegimeSnapshot:
    regime: RegimeState
    allowed_side: str
    confidence: float
    probabilities: dict[RegimeState, float]
    state_mean_returns: dict[RegimeState, float]
    state_volatility: dict[RegimeState, float]
    garch_volatility: float
    garch_volatility_ratio: float
    fit_status: str
    reason: str

In [ ]:
def compute_regime_frame(
    candles_15m: pd.DataFrame,
    lookback: int = 240,
    min_history: int = 80,
) -> pd.DataFrame:
    # Pull out a clean close-price series from the candle DataFrame.
    # This also makes sure the index is a sorted UTC DatetimeIndex.
    closes = _extract_close_series(candles_15m)

    # If we have no usable close data, return an empty frame immediately.
    if closes.empty:
        return pd.DataFrame()

    # Restrict the history to the most recent window.
    # We keep at least min_history + 1 prices because returns need one extra price point.
    if lookback > 0:
        closes = closes.tail(max(lookback, min_history + 1))

    # Compute 15-minute log returns:
    # r_t = log(P_t) - log(P_{t-1})
    returns = np.log(closes).diff().dropna()

    # If there is not enough return history to fit the model, use the fallback classifier.
    if len(returns) < min_history:
        snapshot = _fallback_snapshot(returns, fit_status="insufficient_history")
        return _snapshot_to_frame(returns.index, snapshot)

    try:
        # ------------------------------------------------------------
        # STEP 1: Fit a GARCH(1,1) model to estimate time-varying variance
        # ------------------------------------------------------------
        # This produces conditional variance sigma_t^2 for each return.
        # The purpose is to account for volatility clustering before fitting regimes.
        garch_variance = _fit_garch11_variance(returns)

        # ------------------------------------------------------------
        # STEP 2: Standardize returns by volatility
        # ------------------------------------------------------------
        # z_t = r_t / sigma_t
        # where sigma_t = sqrt(garch_variance_t)
        #
        # This puts returns into "volatility units" so that the Markov model
        # sees direction more cleanly instead of getting overly influenced by
        # high-volatility periods.
        standardized_returns = (
            returns / np.sqrt(garch_variance)
        ).replace([np.inf, -np.inf], np.nan).dropna()

        # If standardization caused too much data loss, fall back.
        if len(standardized_returns) < min_history:
            snapshot = _fallback_snapshot(returns, fit_status="garch_standardization_failed")
            return _snapshot_to_frame(returns.index, snapshot)

        # ------------------------------------------------------------
        # STEP 3: Fit a 3-state Markov switching regression
        # ------------------------------------------------------------
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", ConvergenceWarning)

            # Fit a 3-regime Markov switching model on standardized returns.
            # trend="c" means each regime gets its own intercept / mean.
            # switching_variance=True means each regime can also have its own variance.
            model = MarkovRegression(
                standardized_returns,
                k_regimes=3,
                trend="c",
                switching_variance=True,
            )

            # Fit the model.
            # maxiter controls optimization effort.
            # em_iter does a few EM warm-up iterations to help convergence.
            result = model.fit(disp=False, maxiter=100, em_iter=5)

        # ------------------------------------------------------------
        # STEP 4: Get filtered regime probabilities
        # ------------------------------------------------------------
        # This gives, for each time t and each raw hidden state j:
        # P(S_t = j | information up to time t)
        probabilities = result.filtered_marginal_probabilities

        # Sometimes statsmodels may not return a DataFrame, so force it into one.
        #This is an area where we could have some problems if the model to fit
        if not isinstance(probabilities, pd.DataFrame):
            probabilities = pd.DataFrame(
                probabilities,
                index=standardized_returns.index,
                columns=[0, 1, 2],
            )

        # ------------------------------------------------------------
        # STEP 5: Label the raw states as bull / neutral / bear: Designed by chatgpt
        # ------------------------------------------------------------
        # The Markov model itself only knows states 0, 1, 2.
        # We map them into economic labels based on weighted mean returns.
        mapping = _label_regimes(
            returns.loc[standardized_returns.index],
            probabilities,
            garch_variance.loc[standardized_returns.index],
        )

        # Build a labeled output frame indexed by time.
        labeled = pd.DataFrame(index=standardized_returns.index)

        # These dictionaries will store the weighted average return and volatility
        # for each labeled regime.
        mean_map = {}
        vol_map = {}

        # Loop through each raw Markov state and convert it to bull/neutral/bear.
        for raw_state, label in mapping.items():
            # Compute weighted statistics for this state:
            # - weighted mean return
            # - weighted average volatility
            mean_return, vol_value = _weighted_state_stats(
                returns.loc[standardized_returns.index],
                garch_variance.loc[standardized_returns.index],
                probabilities[raw_state],
            )

            # Save the probability time series for this labeled regime.
            labeled[f"{label}_prob"] = probabilities[raw_state].astype(float)

            # Save the summary regime stats.
            mean_map[label] = mean_return
            vol_map[label] = vol_value

        # Ensure all three labels exist, even if one somehow was not assigned.
        # Also attach the estimated state mean return and volatility for each regime.
        for label in REGIME_TO_SIDE:
            if f"{label}_prob" not in labeled:
                labeled[f"{label}_prob"] = 0.0
            labeled[f"{label}_mean_return"] = float(mean_map.get(label, 0.0))
            labeled[f"{label}_volatility"] = float(vol_map.get(label, 0.0))

        # ------------------------------------------------------------
        # STEP 6: Choose the most likely regime at each time
        # ------------------------------------------------------------
        ordered = ["bull", "neutral", "bear"]
        regime_prob_frame = labeled[[f"{name}_prob" for name in ordered]]

        # Current regime = the label with the highest filtered probability.
        labeled["regime"] = regime_prob_frame.idxmax(axis=1).str.replace("_prob", "", regex=False)

        # Confidence = the highest regime probability.
        labeled["confidence"] = regime_prob_frame.max(axis=1).astype(float)

        # ------------------------------------------------------------
        # STEP 7: Add GARCH volatility diagnostics
        # ------------------------------------------------------------
        # Current conditional volatility estimate:
        labeled["garch_volatility"] = np.sqrt(
            garch_variance.loc[standardized_returns.index]
        ).astype(float)

        # Rolling baseline volatility over the last 32 bars.
        # min_periods=8 prevents the early values from all being NaN.
        rolling_baseline = labeled["garch_volatility"].rolling(32, min_periods=8).mean()

        # If rolling baseline is missing early on, fall back to the average volatility.
        baseline_fallback = float(labeled["garch_volatility"].mean()) if not labeled.empty else 1.0
        rolling_baseline = rolling_baseline.fillna(baseline_fallback).replace(0.0, np.nan)

        # Volatility ratio = current volatility / recent baseline volatility.
        # > 1 means current volatility is elevated relative to recent history.
        labeled["garch_volatility_ratio"] = (
            labeled["garch_volatility"] / rolling_baseline.fillna(baseline_fallback)
        ).replace([np.inf, -np.inf], np.nan).fillna(1.0)

        # ------------------------------------------------------------
        # STEP 8: Apply hand-tuned guardrails
        # ------------------------------------------------------------
        # If confidence is too low, we do not trust the directional regime.
        low_conf_mask = labeled["confidence"] < 0.45

        # If the regime label and the estimated mean drift disagree, demote to neutral.
        weak_drift_mask = labeled["regime"].eq("bull") & (labeled["bull_mean_return"] <= 0)
        weak_drift_mask |= labeled["regime"].eq("bear") & (labeled["bear_mean_return"] >= 0)

        # Demote uncertain or sign-inconsistent calls to neutral.
        labeled.loc[low_conf_mask | weak_drift_mask, "regime"] = "neutral"

        # Map regime to trading side.
        labeled["allowed_side"] = labeled["regime"].map(REGIME_TO_SIDE).fillna("NO_TRADE")

        # Build a readable text explanation for each timestamp.
        labeled["fit_status"] = "fit"
        labeled["reason"] = labeled.apply(_build_reason, axis=1)

        return labeled

    except Exception:
        # If anything fails in the GARCH + Markov fit, fall back to a simpler heuristic.
        snapshot = _fallback_snapshot(returns, fit_status="fallback")
        return _snapshot_to_frame(returns.index, snapshot)

In [ ]:
#Fitting the MSGARCH, snapshot gives us the last time series value to calculate a regime flip

def fit_markov_garch_regime(
    candles_15m: pd.DataFrame,
    lookback: int = 240,
    min_history: int = 80,
) -> RegimeSnapshot:
    regime_frame = compute_regime_frame(
        candles_15m=candles_15m,
        lookback=lookback,
        min_history=min_history,
    )
    return snapshot_from_regime_frame(regime_frame)